# Individual versus pooled training

This experiment asks whether adding the other source datasets improves prediction on the same held-out observations. It reuses existing pooled predictions and outer splits. Each source-only model is independently tuned using the original candidate grid and the source-restricted inner splits. It does not test transfer to a completely unseen dataset.

Place this notebook and `pooling_comparison.py` together in the repository’s `notebooks` folder. Select the same Python kernel used for the completed experiments. Keep the three component CSVs and combined CSV in `Dataset`.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks"))
from pooling_comparison import run_pooling_comparison


## Configuration

The existing result folders must contain `metadata.json`, `splits.json`, and `predictions.csv`. You may point to the corresponding ZIP instead. `MODELS = None` evaluates all 13 original families. This involves new source-only fits across three datasets and both targets, so allow substantial time. Completed tasks are saved and reused if you rerun with identical settings and environment.

In [ ]:
DATA = ROOT / "Dataset/PVT_1222_valid_gg.csv"
SOURCES = {
    "Dataset_A": ROOT / "Dataset/PVT_0160_valid_gg.csv",
    "Dataset_B": ROOT / "Dataset/PVT_0283_valid_gg.csv",
    "Dataset_C": ROOT / "Dataset/PVT_0779_valid_gg.csv",
}
POOLED_RUNS = {
    "Pb": ROOT / "results/PVT_1222_Pb_all_models",
    "Bob": ROOT / "results/PVT_1222_Bob_all_models",
}
OUTPUT_ROOT = ROOT / "results/matched_pooling"
TARGETS = ("Pb", "Bob")
MODELS = None

for path in [DATA, *SOURCES.values(), *(POOLED_RUNS[t] for t in TARGETS)]:
    if not path.exists():
        raise FileNotFoundError(path)
print("Input paths found. Ready to run.")


In [ ]:
comparisons = {}
for target in TARGETS:
    comparisons[target] = run_pooling_comparison(
        DATA, SOURCES, POOLED_RUNS[target], OUTPUT_ROOT / target, models=MODELS
    )
    print(f"\n{target}: completed")
    display(comparisons[target])


## Reading the output

`individual` is RMSE from source-only training. `pooled` is RMSE from the existing pooled training, evaluated on exactly the same observations. Positive `pooled_rmse_improvement_percent` means pooling reduced RMSE; negative means it increased RMSE. These are scores over all held-out predictions within a source, not averages of fold RMSEs. They are descriptive comparisons and do not establish statistical significance.

Each target output contains predictions, source and fold metrics, source membership, split records, tuning searches, selected parameters, fit audits, and a run manifest with hashes and dependency versions. Keep outputs private: predictions contain measured targets. Do not change tuning grids based on these evaluation results without documenting a new analysis.

To continue after an interruption, rerun the configuration and experiment cells with unchanged settings. To change models or environment, choose a fresh `OUTPUT_ROOT`.